# AL Test Generator — Agente RAG híbrido para Business Central

Este notebook recorre paso a paso todo lo que hace el agente: desde el libro de testing en AL hasta la generación de un codeunit de tests completo.

## Arquitectura general

```
libro_al_testing.md          guarantees/src/*.al       guarantees-test/**/*.al
       │                             │                           │
       ▼                             ▼                           ▼
  parser.py                    al_parser.py               al_parser.py
  chunker.py                  (app_code)              (company_example)
       │                             │                           │
       └─────────────────────────────┴───────────────────────────┘
                                     │
                                     ▼
                             embedder.py (BGE-M3)
                                     │
                                     ▼
                             ChromaDB vectorstore
                                     │
                    ┌────────────────┴─────────────────┐
                    ▼                                   ▼
              Python retrieval                    LLM (hasta 2 tool calls extra)
           (RAG + fichero fuente                        │
            + test de referencia)                       │
                    └────────────────┬─────────────────┘
                                     ▼
                               Prompt compacto
                                     │
                                     ▼
                               LLM → codeunit AL
```

## 1. Configuración del entorno

In [1]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Añadir la raíz del proyecto al sys.path para poder importar 'src'
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Cargar claves de API y configuración desde .env
load_dotenv(ROOT_DIR / ".env")

# Evitar errores de conexión si ya tienes los modelos en local
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"✓ Entorno configurado. Proyecto en: {ROOT_DIR}")

✓ Entorno configurado. Proyecto en: C:\Users\mscarafu\TFGLaberit


## 2. Paso 1 — Parseo del libro de testing en AL

Usamos el módulo `src.parser` para convertir el Markdown en secciones tipadas.

In [2]:
from src.parser import parse_markdown
from collections import Counter

LIBRO_PATH = ROOT_DIR / "Data" / "libro_al_testing.md"

with open(LIBRO_PATH, encoding="utf-8") as f:
    md_text = f.read()

sections = parse_markdown(md_text)
type_counts = Counter(s.content_type for s in sections)

print(f'Total secciones: {len(sections)}')
for t, c in sorted(type_counts.items()):
    print(f'  {t:15s}: {c}')

Total secciones: 100
  concept        : 59
  reference      : 1
  summary        : 5
  test_example   : 35


## 3. Paso 2 — Chunking del libro

Dividimos las secciones en fragmentos (chunks) compatibles con LangChain usando `src.chunker`.

In [3]:
from src.chunker import sections_to_documents

book_docs = sections_to_documents(sections)
print(f"Total chunks del libro: {len(book_docs)}")

ModuleNotFoundError: No module named 'langchain.schema'

## 4. Paso 3 — Parseo de los archivos AL del proyecto

Usamos `src.al_parser` para leer los ficheros `.al` del proyecto de Business Central.

In [ ]:
from src.al_parser import parse_al_directory
from src.project_config import ProjectConfig

config = ProjectConfig.default()
al_docs = parse_al_directory(src_dirs=config.src_dirs, test_dirs=config.test_dirs)

print(f"Total chunks AL del proyecto: {len(al_docs)}")

## 5. Paso 4 — Ingesta en el Vectorstore (ChromaDB)

Usamos `src.embedder` para generar los embeddings e ingestar los documentos.

In [ ]:
from src.embedder import load_embeddings, ingest_documents, DEFAULT_PERSIST

embeddings = load_embeddings(device="cpu")
all_chunks = book_docs + al_docs

# NOTA: Descomenta para ingestar por primera vez
# vectorstore = ingest_documents(all_chunks, embeddings, persist_dir=DEFAULT_PERSIST, overwrite=True)

print(f"Embeddings y Vectorstore listos.")

## 6. Paso 5 — Generación con el Agente

Finalmente, inicializamos el `ALTestAgent` de `src.agent` y generamos un test.

In [ ]:
from src.agent import ALTestAgent

agent = ALTestAgent(
    provider = "claude", # Ajusta según tu .env
    persist_dir = DEFAULT_PERSIST
)

QUERY = "Generate a test codeunit for the PostGuarantee procedure in DTNGARManagement"
result = agent.generate(QUERY, show_context=True)

print("\n" + "="*70)
print("CÓDIGO AL GENERADO")
print("="*70 + "\n")
print(result)